# Brain-behavior correlations

Correlates per-subject behavioral/cognitive measures against per-subject brain measures, all
joined on `participant_id` (+ noise level, where both sides have one):

1. **Univariate beta vs. accuracy/RT** (per ROI x noise level) -- `group_level_all_ROI.ipynb`'s
   `roi_df_long` against `behavior_analysis.ipynb`'s `behavior_summary_by-noise-level.csv`.
2. **RSA syllable-model fit vs. accuracy** (per ROI, noiselevel-Q) -- does how strongly an ROI
   represents syllable identity relate to task accuracy in that same (quiet) condition.
3. **RSA syllable-model fit vs. CTOPP phonological awareness** (per ROI, noiselevel-Q) -- CTOPP
   is a more specific, theoretically-motivated correlate of *representational* content than of
   raw activation magnitude (the compelling version of this analysis: does an individual's
   phonological-awareness score predict how strongly their brain represents syllable identity).
4. **Univariate beta vs. CTOPP** (per ROI x noise level) -- same idea as (1), for completeness.

All correlations are Spearman (rank-based -- robust to the non-normality/outliers typical of
psychometric scores and small samples), computed **separately per group** (CWS/CWNS pooled
correlations risk detecting a spurious relationship driven by group mean differences rather than
real within-group brain-behavior coupling), with FDR correction applied **within each group's own
family** -- matching the same group-stratified convention already applied to the RSA and
univariate ROI statistics elsewhere in this pipeline (2026-08-13). CWS's much smaller N means any
CWS-only correlation here should be treated as exploratory regardless of p-value -- a handful of
subjects can drive a correlation coefficient.

This notebook only *reads* already-cached outputs from the other three notebooks (behavior, ROI,
RSA) -- run those first if their outputs are stale or missing.

In [ ]:
import os
import pickle

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

In [ ]:
bidsroot = os.path.join('/ix1/bchandrasekaran/krs228/data/',
                        'SSP/',
                        'data_bids')
nilearn_dir = os.path.join(bidsroot, 'derivatives', 'nilearn')
glmsingle_dir = os.path.join(bidsroot, 'derivatives', 'glmsingle')
behavior_out_dir = os.path.join(bidsroot, 'derivatives', 'behavior')

FWHM = 6.00
RSA_NOISE_LEVEL_TAG = 'Q'  # matches report/generate_report.py's convention -- the noise level
                           # with the RSA pipeline's main, best-established results (L-STGp/syllable)

group_out_dir = os.path.join(nilearn_dir, 'group_fwhm-%.02f' % FWHM)
rsa_out_dir = os.path.join(glmsingle_dir, 'rsa-group_glmsingle')

out_dir = os.path.join(behavior_out_dir, 'brain_behavior_correlations')
os.makedirs(out_dir, exist_ok=True)

GROUP_ORDER = ['CWS', 'CWNS']
GROUP_PALETTE = {'CWNS': '#009E73', 'CWS': '#CC79A7'}  # same palette as every other group-level notebook

## Load the three cached inputs

Noise-level labels are **not consistently cased across these files** -- `roi_df_long`'s `SNR`
column uses lowercase (`'q'`, matching `univariate_fmri/group_level_all_ROI.ipynb`'s
`contrast_list`), while `behavior_summary_df`'s `noise_level` and RSA's `NOISE_LEVEL_TAGS` both
use uppercase (`'Q'`). A naive join on these columns directly would silently drop every quiet-
condition row (no match found) rather than error -- normalized to uppercase before any join
below.

In [ ]:
participants_fpath = os.path.join(bidsroot, 'participants.tsv')
participants_df = pd.read_csv(participants_fpath, sep='\t')
participants_df = participants_df.set_index('participant_id')

behavior_summary_df = pd.read_csv(os.path.join(behavior_out_dir, 'behavior_summary_by-noise-level.csv'))
behavior_summary_df['noise_level'] = behavior_summary_df['noise_level'].astype(str).str.upper()

with open(os.path.join(group_out_dir, 'roi_df_long.pkl'), 'rb') as f:
    roi_df_long = pickle.load(f)
roi_df_long = roi_df_long.copy()
roi_df_long['noise_level'] = roi_df_long['SNR'].astype(str).str.upper()

rsa_model_fit_fpath = os.path.join(rsa_out_dir, f'model_fit_scalars_noiselevel-{RSA_NOISE_LEVEL_TAG}.csv')
model_fit_df = pd.read_csv(rsa_model_fit_fpath)

print(f'behavior_summary_df: {len(behavior_summary_df)} rows, '
     f'{behavior_summary_df.participant_id.nunique()} subjects')
print(f'roi_df_long: {len(roi_df_long)} rows, {roi_df_long.participant_id.nunique()} subjects, '
     f'{roi_df_long.region_hemi.nunique()} ROIs')
print(f'model_fit_df (noiselevel-{RSA_NOISE_LEVEL_TAG}): {len(model_fit_df)} rows, '
     f'{model_fit_df.participant_id.nunique()} subjects, {model_fit_df.ROI.nunique()} ROIs, '
     f'models: {sorted(model_fit_df.model.unique())}')

## Reusable correlation + group-stratified FDR helper

In [ ]:
def compute_correlations(df, brain_col, behavior_col, groupby_cols, min_n=4):
    """Spearman correlation between brain_col and behavior_col, computed separately for every
    combination of groupby_cols (must include 'group'), with FDR correction applied WITHIN each
    group's own family -- not pooled across groups, matching the same group-stratified
    convention already applied to the RSA/univariate ROI statistics elsewhere in this pipeline.
    A cell with fewer than min_n paired (non-NaN) observations is skipped rather than producing
    a degenerate/unstable correlation estimate.
    """
    assert 'group' in groupby_cols, "groupby_cols must include 'group' for per-group FDR"
    records = []
    for keys, g in df.groupby(groupby_cols, observed=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        g = g.dropna(subset=[brain_col, behavior_col])
        if len(g) < min_n:
            continue
        rho, p = spearmanr(g[brain_col], g[behavior_col])
        rec = dict(zip(groupby_cols, keys))
        rec.update({'n': len(g), 'rho': rho, 'p': p})
        records.append(rec)

    result_df = pd.DataFrame(records)
    if len(result_df) > 0:
        result_df['p_fdr'] = result_df.groupby('group')['p'].transform(
            lambda p: multipletests(p, method='fdr_bh')[1])
    return result_df


def summarize_hits(result_df, label):
    if result_df is None or len(result_df) == 0:
        print(f'{label}: no cells had enough paired observations to test.')
        return
    for group_name in GROUP_ORDER:
        group_df = result_df[result_df.group == group_name]
        if len(group_df) == 0:
            continue
        sig = group_df[group_df.p_fdr < 0.05].sort_values('p_fdr')
        print(f'{label} -- {group_name}: {len(sig)}/{len(group_df)} FDR-significant '
             f'(smallest p_fdr={group_df.p_fdr.min():.3f})')
        if len(sig) > 0:
            print(sig.to_string(index=False))

## 1. Univariate beta vs. accuracy / RT (per ROI x noise level)

In [ ]:
univariate_behavior_df = roi_df_long.merge(
    behavior_summary_df[['participant_id', 'noise_level', 'accuracy', 'rt_mean_correct']],
    on=['participant_id', 'noise_level'], how='inner')
print(f'{len(univariate_behavior_df)} matched (subject, ROI, noise level) rows '
     f'(of {len(roi_df_long)} in roi_df_long)')

univ_accuracy_corr_df = compute_correlations(
    univariate_behavior_df, brain_col='beta', behavior_col='accuracy',
    groupby_cols=['group', 'region_hemi', 'noise_level'])
summarize_hits(univ_accuracy_corr_df, 'Univariate beta vs. accuracy')

univ_rt_corr_df = compute_correlations(
    univariate_behavior_df, brain_col='beta', behavior_col='rt_mean_correct',
    groupby_cols=['group', 'region_hemi', 'noise_level'])
summarize_hits(univ_rt_corr_df, 'Univariate beta vs. RT (correct trials)')

univ_accuracy_corr_df.to_csv(os.path.join(out_dir, 'univariate_vs_accuracy.csv'), index=False)
univ_rt_corr_df.to_csv(os.path.join(out_dir, 'univariate_vs_rt.csv'), index=False)

## 2. RSA syllable-model fit vs. accuracy (per ROI, noiselevel-Q)

In [ ]:
syllable_fit_df = model_fit_df[model_fit_df.model == 'syllable']
accuracy_q_df = behavior_summary_df[behavior_summary_df.noise_level == RSA_NOISE_LEVEL_TAG]

rsa_behavior_df = syllable_fit_df.merge(
    accuracy_q_df[['participant_id', 'accuracy', 'rt_mean_correct']],
    on='participant_id', how='inner')
print(f'{len(rsa_behavior_df)} matched (subject, ROI) rows '
     f'(of {len(syllable_fit_df)} syllable-model rows)')

rsa_accuracy_corr_df = compute_correlations(
    rsa_behavior_df, brain_col='fit', behavior_col='accuracy', groupby_cols=['group', 'ROI'])
summarize_hits(rsa_accuracy_corr_df, f'RSA syllable-fit vs. accuracy (noiselevel-{RSA_NOISE_LEVEL_TAG})')

rsa_accuracy_corr_df.to_csv(
    os.path.join(out_dir, f'rsa-syllable_vs_accuracy_noiselevel-{RSA_NOISE_LEVEL_TAG}.csv'), index=False)

## 3. RSA syllable-model fit vs. CTOPP phonological awareness (per ROI, noiselevel-Q)

The more theoretically specific brain-behavior test: does an individual's phonological-awareness
score predict how strongly their brain represents syllable identity, rather than just how
accurately they perform the task.

In [ ]:
rsa_ctopp_df = syllable_fit_df.merge(
    participants_df[['ctopp_phon_awareness']], left_on='participant_id', right_index=True, how='inner')
print(f'{len(rsa_ctopp_df)} matched (subject, ROI) rows with a CTOPP score '
     f'(of {len(syllable_fit_df)} syllable-model rows)')

rsa_ctopp_corr_df = compute_correlations(
    rsa_ctopp_df, brain_col='fit', behavior_col='ctopp_phon_awareness', groupby_cols=['group', 'ROI'])
summarize_hits(rsa_ctopp_corr_df, f'RSA syllable-fit vs. CTOPP (noiselevel-{RSA_NOISE_LEVEL_TAG})')

rsa_ctopp_corr_df.to_csv(
    os.path.join(out_dir, f'rsa-syllable_vs_ctopp_noiselevel-{RSA_NOISE_LEVEL_TAG}.csv'), index=False)

## 4. Univariate beta vs. CTOPP (per ROI x noise level)

Same idea as section 1, swapping accuracy/RT for CTOPP. CTOPP is a single subject-level value
(not noise-level-specific), so it's broadcast across every noise level for a given subject before
correlating -- a real, if less specific, alternative to the RSA version above.

In [ ]:
univariate_ctopp_df = roi_df_long.merge(
    participants_df[['ctopp_phon_awareness']], left_on='participant_id', right_index=True, how='inner')
print(f'{len(univariate_ctopp_df)} matched (subject, ROI, noise level) rows with a CTOPP score')

univ_ctopp_corr_df = compute_correlations(
    univariate_ctopp_df, brain_col='beta', behavior_col='ctopp_phon_awareness',
    groupby_cols=['group', 'region_hemi', 'noise_level'])
summarize_hits(univ_ctopp_corr_df, 'Univariate beta vs. CTOPP')

univ_ctopp_corr_df.to_csv(os.path.join(out_dir, 'univariate_vs_ctopp.csv'), index=False)

## Scatter plot helper (for inspecting any specific hit above)

In [ ]:
def plot_correlation_scatter(df, brain_col, behavior_col, group_name, subset_label, title):
    """subset_label identifies which row of df to plot -- e.g. a specific (region_hemi,
    noise_level) or ROI -- already filtered to a single group/cell's data before calling.
    """
    fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=150)
    sns.regplot(data=df, x=behavior_col, y=brain_col, ax=ax,
               color=GROUP_PALETTE[group_name], scatter_kws={'s': 25, 'alpha': 0.8})
    rho, p = spearmanr(df[behavior_col], df[brain_col])
    ax.set_title(f'{title}\nn={len(df)}, Spearman rho={rho:.2f}, p={p:.3f}')
    fig.tight_layout()
    sns.despine(ax=ax)
    return fig


# Example usage once real hits exist, e.g.:
# hit_df = rsa_ctopp_df[(rsa_ctopp_df.group == 'CWNS') & (rsa_ctopp_df.ROI == 'L-STGp')]
# plot_correlation_scatter(hit_df, 'fit', 'ctopp_phon_awareness', 'CWNS', 'L-STGp',
#                          'L-STGp syllable-fit vs. CTOPP')

## Summary

In [ ]:
print('=== Brain-behavior correlation summary ===')
summarize_hits(univ_accuracy_corr_df, 'Univariate beta vs. accuracy')
summarize_hits(univ_rt_corr_df, 'Univariate beta vs. RT')
summarize_hits(rsa_accuracy_corr_df, 'RSA syllable-fit vs. accuracy')
summarize_hits(rsa_ctopp_corr_df, 'RSA syllable-fit vs. CTOPP')
summarize_hits(univ_ctopp_corr_df, 'Univariate beta vs. CTOPP')
print(f'\nAll correlation tables saved to {out_dir}')